In [16]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sentence_transformers import SentenceTransformer
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score
from tqdm.auto import tqdm
from catboost import Pool, cv


from sklearn.metrics import f1_score, make_scorer

tqdm.pandas()

features_path = Path("data/")

In [3]:
def preprocess_dataset(df: pd.DataFrame, embedding_model: SentenceTransformer) -> pd.DataFrame:
    df = df.groupby("cell_type").get_group("code")[["text", "primary_label"]]
    df["text"] = df.text.apply(lambda x: "\n".join(x))

    embeddings = embedding_model.encode(df.text.tolist(), show_progress_bar=True)
    df["embedding"] = list(embeddings)
    return df

In [4]:
embedding_model = SentenceTransformer("mchochlov/codebert-base-cd-ft")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.51k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [5]:
train_features = pd.read_pickle(features_path / "train_features.pkl")
train_features.index = range(train_features.shape[0])
train_features.fillna(0, inplace=True)


df_train = preprocess_dataset(train_features, embedding_model)
df_train.head()

Batches:   0%|          | 0/183 [00:00<?, ?it/s]

,text,primary_label,embedding
0,import pandas as pd\nimport numpy as np,helper_functions,"[-0.05722573, 0.42762598, 0.52219087, -0.07709..."
1,"l_cols = ['user_id','movie_id','rating']\nr_co...",load_data,"[-0.13790527, 0.8422397, 0.20092261, -0.275311..."
2,l.head(),data_exploration,"[-0.10054576, 0.22681685, 0.39339706, 0.316472..."
3,r.head(),data_exploration,"[-0.16078387, 0.3733044, 0.4019712, 0.2852123,..."
4,"movies = pd.merge(l,r)",data_preprocessing,"[-0.2499569, 0.4617576, -0.091905005, 0.233743..."


In [6]:
validation_features = pd.read_pickle(features_path / "validation_features.pkl")
validation_features.index = range(validation_features.shape[0])
validation_features.fillna(0, inplace=True)

test_features = pd.read_pickle(features_path / "test_features.pkl")
test_features.index = range(test_features.shape[0])
test_features.fillna(0, inplace=True)

df_val = preprocess_dataset(validation_features, embedding_model)
df_test = preprocess_dataset(test_features, embedding_model)

Batches:   0%|          | 0/61 [00:00<?, ?it/s]

Batches:   0%|          | 0/60 [00:00<?, ?it/s]

In [7]:
for df, name in [(df_train, "train"), (df_val, "val"), (df_test, "test")]:
    df.to_pickle(f"data/processed/{name}.pkl")

In [8]:
df_train = pd.read_pickle("data/processed/train.pkl")
df_val = pd.read_pickle("data/processed/val.pkl")
df_test = pd.read_pickle("data/processed/test.pkl")

# Training (no hyperparameters tuning)

In [9]:
X_train, y_train = df_train["embedding"].tolist(), df_train["primary_label"].tolist()
X_val, y_val = df_val["embedding"].tolist(), df_val["primary_label"].tolist()

In [17]:
cb_model = CatBoostClassifier(
    loss_function="MultiClass",
    eval_metric="TotalF1",
    iterations=400,
    random_seed=42,
)

train_pool = Pool(X_train, y_train)

# Perform cross-validation
cv_results = cv(train_pool, cb_model.get_params(), fold_count=5)

# Get the best iteration
best_iteration = cv_results["iterations"]

# Train the model with the best iteration
# cb_model = CatBoostClassifier(iterations=best_iteration)
# cb_model.fit(X_train, y_train, eval_set=(X_val, y_val),
#     use_best_model=True,
#     plot=True,
#     verbose=False,)

Training on fold [0/5]
0:	learn: 0.4347565	test: 0.4103874	best: 0.4103874 (0)	total: 113ms	remaining: 44.9s
1:	learn: 0.4571111	test: 0.4258579	best: 0.4258579 (1)	total: 213ms	remaining: 42.4s
2:	learn: 0.4843294	test: 0.4582577	best: 0.4582577 (2)	total: 320ms	remaining: 42.4s
3:	learn: 0.4984056	test: 0.4767972	best: 0.4767972 (3)	total: 431ms	remaining: 42.7s
4:	learn: 0.5153565	test: 0.5052319	best: 0.5052319 (4)	total: 530ms	remaining: 41.8s
5:	learn: 0.5343631	test: 0.5141147	best: 0.5141147 (5)	total: 653ms	remaining: 42.9s
6:	learn: 0.5318575	test: 0.5105951	best: 0.5141147 (5)	total: 749ms	remaining: 42.1s
7:	learn: 0.5358685	test: 0.5161137	best: 0.5161137 (7)	total: 856ms	remaining: 41.9s
8:	learn: 0.5370039	test: 0.5266222	best: 0.5266222 (8)	total: 952ms	remaining: 41.4s
9:	learn: 0.5430947	test: 0.5282843	best: 0.5282843 (9)	total: 1.05s	remaining: 41s
10:	learn: 0.5459960	test: 0.5227289	best: 0.5282843 (9)	total: 1.16s	remaining: 40.9s
11:	learn: 0.5487929	test: 0.532

KeyboardInterrupt: 

In [39]:
eval_metric = "TotalF1:average=Weighted"
params = {
    "loss_function": "MultiClass",
    "eval_metric": eval_metric,
    "iterations": 400,
    "random_seed": 42,
    "learning_rate": 0.2,
}

cv_data = cv(
    params=params,
    pool=Pool(X_train, label=y_train),
    fold_count=5,
    shuffle=True,
    partition_random_seed=0,
    plot=True,
    stratified=True,
    verbose=False,
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

Training on fold [0/5]

bestTest = 0.7647326635
bestIteration = 363

Training on fold [1/5]

bestTest = 0.7733629881
bestIteration = 347

Training on fold [2/5]

bestTest = 0.7926698468
bestIteration = 277

Training on fold [3/5]

bestTest = 0.7674109572
bestIteration = 387

Training on fold [4/5]

bestTest = 0.7641975821
bestIteration = 343



In [40]:
eval_metric = "TotalF1:average=Weighted"
best_value = np.max(cv_data[f"test-{eval_metric}-mean"])
best_iter = np.argmax(cv_data[f"test-{eval_metric}-mean"])
print(
    "Best validation {} score, stratified: {:.4f}+/-{:.3f} on step {}".format(
        eval_metric, best_value, cv_data[f"test-{eval_metric}-std"][best_iter], best_iter
    )
)

Best validation TotalF1:average=Weighted score, stratified: 0.7703+/-0.011 on step 387


In [41]:
cb_model = CatBoostClassifier(**params)

cb_model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    use_best_model=True,
    plot=True,
    verbose=False,
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

In [42]:
cb_model.save_model("weights/catboost_model")

# Evaluation

## Test F1 Score

In [43]:
for set_name, evaluation_set in [("val", df_val), ("test", df_test)]:
    y_pred = cb_model.predict(evaluation_set["embedding"].tolist())
    y_true = evaluation_set.primary_label
    f1 = f1_score(y_pred, y_true, average="weighted")
    print(set_name, f1)

val 0.7255882476870177
test 0.7210395903097851


# Load and inference

In [45]:
inference_model = CatBoostClassifier().load_model("weights/catboost_model")

In [46]:
for set_name, evaluation_set in [("val", df_val), ("test", df_test)]:
    y_pred = inference_model.predict(evaluation_set["embedding"].tolist())
    y_true = evaluation_set.primary_label
    f1 = f1_score(y_pred, y_true, average="weighted")
    print(set_name, f1)

val 0.7255882476870177
test 0.7210395903097851
